## Install Libraries

Runs candidate retrieval (architecture §5.2/§5.3): load the KB data
(`kb.csv`) and the single-file KB artifact built by `build_KB.ipynb`, load
the gold BEL data from Google Drive, run the configured GRF retrieval pipeline,
and save every mention's candidates to **one** JSONL file.

This notebook reports **candidate-retrieval** metrics only (R@1/R@5). The
end-to-end re-ranking/accuracy stage is separate: `reranking.ipynb`.

Sources: `docs/architecture.md`, `paper/reference_abstract.md`,
`source/GRF/README.md`.

Same encoder pin as `source/GRF/requirements.txt`; `transformers` backs
`LLM_BACKEND = "local"` and is unused otherwise. `tqdm` is installed
explicitly because every long stage reports progress.

In [ ]:
%pip install "tqdm>=4.66" "sentence-transformers>=2.7" "transformers>=4.44"

## Import Libraries

Third-party imports only. The `common` and `GRF` packages live inside the
repository clone, so project modules are imported at the end of the
repository-setup stage below.

In [ ]:
from __future__ import annotations

import gc
import os
import subprocess
import sys
import time
from contextlib import contextmanager
from pathlib import Path

from tqdm.auto import tqdm

print("Python:", sys.version.split()[0])
print("tqdm:", __import__("tqdm").__version__)


@contextmanager
def progress_bar(total, desc: str, unit: str = "item"):
    """Drive a tqdm bar from a progress_callback(done, total)."""
    bar = tqdm(total=total, desc=desc, unit=unit)

    def report(done: int, total=None) -> None:
        if total is not None and total != bar.total:
            bar.total = total
        bar.update(done - bar.n)

    try:
        yield report
    finally:
        bar.close()

@contextmanager
def phase_progress():
    """One tqdm bar per pipeline phase, created when that phase first reports.

    ``link_batch`` reports ``(phase, done, total)``; phases run in order, so
    each gets its own bar instead of one bar counting a mixture of work.
    A phase whose ``total`` is None (the encoder batches internally) shows a
    running count rather than a percentage.
    """
    bars = {}

    def report(phase, done, total=None) -> None:
        bar = bars.get(phase)
        if bar is None:
            bar = tqdm(total=total, desc=phase, unit="item", leave=True)
            bars[phase] = bar
        if total is not None and total != bar.total:
            bar.total = total
        bar.update(done - bar.n)

    try:
        yield report
    finally:
        for bar in bars.values():
            bar.close()


## Constants and Configuration

All paths, retrieval parameters and LLM feedback settings live here. The Git
repository and branch are preconfigured.

Drive layout:

- `COMMON_DIR/` — shared root (`DRIVE_ROOT/work/nlp/UMLS/entity_linking`),
- `COMMON_DIR/data/` — shared inputs: `kb.csv`, the `.npz` artifact from
  `build_KB.ipynb`, and the gold BEL JSONL,
- `COMMON_DIR/output/{METHOD}/` — this method's outputs (`METHOD = "grf"`).

`FEEDBACK_TYPES = ()` reproduces the no-GRF baseline; `("def",)`, `("syn",)`,
`("std",)` select single GRF components; `("def", "syn", "std")` is All-in-One
(architecture §6, §7). Retrieval stops at the ranking: no re-ranking runs in this
notebook. `rerank="top1"` is the paper's *Pick Top-1 (không re-rank)* branch
(Bảng 3) and only fills `Response.cui` with the ranking's first entry; `reranking.ipynb`
applies the real re-ranker to the saved candidates.

**Feedback model.** `LLM_BACKEND = "local"` loads a HuggingFace causal LM
directly through `transformers` (`LLM_MODEL_LOCAL`: a Hub id or a local
directory) — no API key, no network at inference time once the weights are
cached. `LLM_BACKEND = "api"` uses the OpenAI-compatible HTTP client and needs
`OPENAI_API_KEY`. Both paths degrade to `Q_0` on failure (architecture §6).

In [ ]:
# ---------------------------------------------------------------------------
# Git repository (REQUIRED: replace the placeholder with your repository URL)
# ---------------------------------------------------------------------------
GIT_REPO_URL = "https://github.com/minhhhduc/bio-linking-.git"
GIT_BRANCH = "main"                                             # TODO
REPO_CLONE_DIR = Path("/content/bio_linking")

# ---------------------------------------------------------------------------
# Google Drive layout
#   COMMON_DIR/                     shared root
#   COMMON_DIR/data/                shared inputs (kb.csv, artifact, gold data)
#   COMMON_DIR/output/{METHOD}/     this method's outputs
# ---------------------------------------------------------------------------
METHOD = "grf"
MOUNT_DRIVE = True
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
COMMON_DIR = DRIVE_ROOT / "work" / "nlp" / "UMLS" / "entity_linking"
DRIVE_DATA_DIR = COMMON_DIR / "data"
OUTPUT_DIR = COMMON_DIR / "output" / METHOD

KB_CSV_PATH = DRIVE_DATA_DIR / "kb.csv"                     # KB data (input)
KB_ARTIFACT_PATH = DRIVE_DATA_DIR / "grf_kb.npz"                # from build_KB.ipynb
GOLD_DATA_PATH = DRIVE_DATA_DIR / "test.jsonl"                  # gold BEL JSONL
CANDIDATES_OUTPUT_PATH = OUTPUT_DIR / "grf_candidates.jsonl"      # single output

# ---------------------------------------------------------------------------
# Encoder (must match the encoder_info stored in the KB artifact)
# ---------------------------------------------------------------------------
ENCODER_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
ENCODER_BATCH_SIZE = 32
DEVICE = "cuda"                # "cuda" (Colab GPU) or "cpu"

# ---------------------------------------------------------------------------
# GRF retrieval parameters (architecture §9)
# ---------------------------------------------------------------------------
FEEDBACK_TYPES = ("def",)       # (), ("def",), ("syn",), ("std",), or all three
N_SYN = 5                       # one of 1, 3, 5, 10
TARGET_KB = "KB"                # KB name used by the `std` instruction
TOP_K_RETRIEVAL = 10
TOP_K_CONCEPT = 5               # 1..10; candidates saved for e2e
FUSION = "vector"               # "vector" (Pipeline 1) or "rrf" (Pipeline 2)
USE_CONTEXT_QUERY = True        # True = mention + context; False = encode(mention) alone
INDEX_KB = None                 # optional source_kb filter; None = all entries

# ---------------------------------------------------------------------------
# Feedback model: local HuggingFace weights (default) or the OpenAI-compatible API
# ---------------------------------------------------------------------------
LLM_BACKEND = "local"           # "local" (transformers) or "api"

# --- "local": loaded through transformers; HF id or a local directory path ---
LLM_MODEL_LOCAL = "Qwen/Qwen3.8-27B"
LLM_LOCAL_DEVICE = "auto"       # "auto", "cuda", "cpu" or "cuda:0"
LLM_LOCAL_DTYPE = "auto"        # "auto", "float16", "bfloat16" or "float32"
LLM_LOCAL_MAX_NEW_TOKENS = 256  # decode budget per feedback call
LLM_LOCAL_TEMPERATURE = 0.0     # 0.0 = greedy (deterministic)
LLM_LOCAL_TRUST_REMOTE_CODE = False
LLM_LOCAL_USE_CHAT_TEMPLATE = True   # False = feed the raw prompt

# --- "api": OpenAI-compatible HTTP endpoint; needs OPENAI_API_KEY ---
LLM_MODEL_API = "gpt-4o"
LLM_API_BASE_URL = ""           # empty -> OPENAI_BASE_URL env var / api.openai.com
LLM_API_TIMEOUT_S = 30.0
# Set the key in this runtime (never commit it):
#   import getpass, os; os.environ["OPENAI_API_KEY"] = getpass.getpass()

LLM_ACTIVE_MODEL = LLM_MODEL_LOCAL if LLM_BACKEND == "local" else LLM_MODEL_API

# ---------------------------------------------------------------------------
# Evaluation / runtime
# ---------------------------------------------------------------------------
MAX_SAMPLES = None              # None = all messages; an int truncates for smoke runs
CACHE_DIR = Path("/content/grf_cache")
HF_HOME = CACHE_DIR / "huggingface"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_HOME))

_START = time.time()


def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - _START:6.1f}s] {message}", flush=True)


def require_configured(value, name: str):
    text = str(value)
    if "REPLACE_ME" in text:
        raise ValueError(f"{name} is still a placeholder: {text!r}. Set it in the constants cell.")
    return value


log("Configuration loaded.")
print(f"Repository   : {GIT_REPO_URL} ({GIT_BRANCH})")
print(f"KB data      : {KB_CSV_PATH}")
print(f"KB artifact  : {KB_ARTIFACT_PATH}")
print(f"Gold data    : {GOLD_DATA_PATH}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Candidates   : {CANDIDATES_OUTPUT_PATH}")
print(f"GRF settings : feedback={FEEDBACK_TYPES} n_syn={N_SYN} fusion={FUSION}")
print(f"Feedback LLM : backend={LLM_BACKEND} model={LLM_ACTIVE_MODEL}")

## Stage 1 — Clone or Update the GRF Repository

Colab starts from an empty runtime, so the notebook clones the code repository
and imports the shared modules from `source/common/` and the GRF modules from
`source/GRF/`. Re-running this cell updates the existing clone instead of
downloading it again.

In [ ]:
repo_url = str(require_configured(GIT_REPO_URL, "GIT_REPO_URL"))


def _git(*args: str) -> None:
    log("git " + " ".join(args))
    try:
        subprocess.run(["git", *args], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"git {args[0]} failed with exit code {exc.returncode}.") from exc


if REPO_CLONE_DIR.exists() and not (REPO_CLONE_DIR / ".git").exists():
    raise RuntimeError(
        f"{REPO_CLONE_DIR} exists but is not a git clone. "
        "Remove it or point REPO_CLONE_DIR elsewhere."
    )

if (REPO_CLONE_DIR / ".git").exists():
    log(f"Repository already present at {REPO_CLONE_DIR}; updating branch {GIT_BRANCH!r}.")
    _git("-C", str(REPO_CLONE_DIR), "fetch", "--depth", "1", "origin", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "checkout", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "pull", "--ff-only", "origin", GIT_BRANCH)
else:
    log(f"Cloning {repo_url} (branch {GIT_BRANCH!r}) into {REPO_CLONE_DIR} ...")
    REPO_CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    _git("clone", "--depth", "1", "--branch", GIT_BRANCH, repo_url, str(REPO_CLONE_DIR))

SOURCE_DIR = REPO_CLONE_DIR / "source"
if not (SOURCE_DIR / "common" / "__init__.py").is_file():
    raise FileNotFoundError(
        f"{SOURCE_DIR / 'common' / '__init__.py'} is missing; check GIT_BRANCH and the repository layout."
    )
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from common.candidates import load_candidates, save_candidates
from common.data import load_bel_jsonl
from common.encoder import SentenceTransformerEncoder
from common.evaluator import accuracy_at_k, metrics_from_responses
from common.index import SINGLE_FILE_FORMAT, VectorIndex
from common.kb import load_concepts_csv

from GRF.config import PipelineConfig
from GRF.evaluator import make_requests
from GRF.feedback import FeedbackGenerator
from GRF.hf_llm import TransformersChatClient
from GRF.llm import OpenAICompatibleClient
from GRF.pipeline import EntityLinker

log(f"Repository ready at {REPO_CLONE_DIR}; common and GRF imported from {SOURCE_DIR}.")
log(f"Single-file artifact format: {SINGLE_FILE_FORMAT}")


def build_llm_client(backend, model):
    """Local transformers model or the OpenAI-compatible API client."""
    if backend == "local":
        return TransformersChatClient(
            model,
            device=LLM_LOCAL_DEVICE,
            dtype=LLM_LOCAL_DTYPE,
            max_new_tokens=LLM_LOCAL_MAX_NEW_TOKENS,
            temperature=LLM_LOCAL_TEMPERATURE,
            trust_remote_code=LLM_LOCAL_TRUST_REMOTE_CODE,
            use_chat_template=LLM_LOCAL_USE_CHAT_TEMPLATE,
        )
    if backend == "api":
        if not os.environ.get("OPENAI_API_KEY"):
            log(
                "WARNING: OPENAI_API_KEY is not set. API calls will fail and "
                "retrieval will degrade to the baseline path."
            )
        return OpenAICompatibleClient(
            model, base_url=LLM_API_BASE_URL or None, timeout_s=LLM_API_TIMEOUT_S
        )
    raise ValueError(f"LLM_BACKEND must be 'local' or 'api', got {backend!r}")


log("LLM factory ready; feedback backend=%s." % LLM_BACKEND)

## Stage 2 — Mount Google Drive and Locate Inputs

The KB data (`kb.csv`), the KB artifact and the gold data live on Drive, not
in the repository. This stage mounts Drive and checks every input path plus the
single candidate output path before any model is loaded.

In [ ]:
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:  # running outside Colab
        raise RuntimeError(
            "Google Colab is required to mount Drive. Run this notebook in Colab, "
            "or set MOUNT_DRIVE = False only if the inputs are already local to the runtime."
        ) from exc
    drive.mount(str(DRIVE_MOUNT_POINT))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

kb_path = Path(require_configured(KB_CSV_PATH, "KB_CSV_PATH"))
artifact_path = Path(require_configured(KB_ARTIFACT_PATH, "KB_ARTIFACT_PATH"))
gold_path = Path(require_configured(GOLD_DATA_PATH, "GOLD_DATA_PATH"))
candidates_path = Path(require_configured(CANDIDATES_OUTPUT_PATH, "CANDIDATES_OUTPUT_PATH"))

for label, path in (
    ("KB data", kb_path),
    ("KB artifact", artifact_path),
    ("Gold data", gold_path),
):
    if not path.is_file():
        raise FileNotFoundError(f"{label} not found: {path}")
    log(f"{label}: {path} ({path.stat().st_size / 1024 ** 2:.2f} MB)")

if candidates_path.suffix.lower() != ".jsonl":
    raise ValueError(
        f"CANDIDATES_OUTPUT_PATH must end in .jsonl, got {candidates_path.name!r}."
    )
log(f"Candidate output: {candidates_path}")

## Stage 3 — Load the KB Artifact and the Encoder

`VectorIndex.load_npz` restores the vectors and metadata from the single file.
The encoder model itself is not stored in the artifact, so this stage loads the
same checkpoint recorded in `encoder_info` and checks the dimensions agree.
With `DEVICE = "cuda"`, `index.use_gpu()` keeps one GPU copy of the vectors:
retrieval is then a batched matrix multiply on the GPU, and only the top-k rows
cross back to CPU.

In [ ]:
log(f"Loading KB artifact {artifact_path} ...")
index = VectorIndex.load_npz(artifact_path)
log(f"Artifact loaded: {len(index):,} entries x {index.vectors.shape[1]} dims.")

stored_encoder = (index.encoder_info or {}).get("model")
if stored_encoder is None:
    log("WARNING: artifact has no encoder_info; assuming ENCODER_MODEL is the same checkpoint.")
elif stored_encoder != ENCODER_MODEL:
    raise ValueError(
        f"Artifact was built with encoder {stored_encoder!r}, but ENCODER_MODEL is "
        f"{ENCODER_MODEL!r}. Set ENCODER_MODEL to the encoder used by build_KB.ipynb."
    )

if DEVICE == "cuda":
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "DEVICE='cuda' but no CUDA device is available. In Colab: "
            "Runtime > Change runtime type > Hardware accelerator > GPU."
        )
    log(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    log("Running on CPU (DEVICE='cpu').")

log(f"Loading encoder {ENCODER_MODEL!r} on {DEVICE} ...")
encoder = SentenceTransformerEncoder(
    ENCODER_MODEL, batch_size=ENCODER_BATCH_SIZE, device=DEVICE
)
if encoder.dim != index.vectors.shape[1]:
    raise ValueError(
        f"Encoder dimension {encoder.dim} does not match artifact dimension "
        f"{index.vectors.shape[1]}."
    )
log(f"Encoder ready: dim={encoder.dim}.")

if DEVICE == "cuda":
    try:
        index.use_gpu(DEVICE)
    except RuntimeError as exc:
        torch.cuda.empty_cache()
        log(f"WARNING: GPU search unavailable ({exc}); using the CPU scan.")
    else:
        log(f"Search device: {index.search_device} (exact cosine on GPU).")
if index.search_device == "cpu":
    log("Search device: cpu (exact cosine on CPU).")

## Stage 4 — Load the KB Data from Google Drive

The KB CSV on Drive is the authoritative source of preferred concept names
used by the LLM re-ranker in `reranking.ipynb`. It is loaded here and cross-checked
against the CUIs stored in the artifact so a stale `grf_kb.npz` fails loudly
instead of mixing an old index with a new KB.

In [ ]:
log(f"Loading KB data from {kb_path} ...")
started = time.time()
with progress_bar(None, "Parsing KB records", "record") as report:
    kb_concepts = load_concepts_csv(kb_path, progress_callback=report)
log(f"Loaded {len(kb_concepts):,} KB concepts in {time.time() - started:.1f}s.")
if not kb_concepts:
    raise ValueError(f"{kb_path} produced no KB concepts.")

concept_names = {concept.cui: concept.name for concept in kb_concepts}
artifact_cuis = set(index.cuis)
kb_cuis = set(concept_names)

missing = artifact_cuis - kb_cuis
if missing:
    raise ValueError(
        f"{len(missing):,} CUIs in the KB artifact are missing from {kb_path}; "
        "the artifact and the KB data do not match. Rebuild it with build_KB.ipynb."
    )
extra = kb_cuis - artifact_cuis
if extra:
    log(
        f"WARNING: {len(extra):,} KB CUIs are absent from the artifact "
        "(the artifact may be stale or filtered)."
    )
log(f"Concept names ready: {len(concept_names):,} CUIs from the KB data.")

## Stage 5 — Load and Validate the Gold Data

`load_bel_jsonl` validates structure and token spans. Structural problems are
reported as `DataIssue` entries; span issues never drop a sample. All samples
are used for candidate retrieval. The loader reports each record through
`progress_callback`, which drives the bar.

In [ ]:
log(f"Loading gold BEL data from {gold_path} ...")
started = time.time()
with progress_bar(None, "Loading gold records", "record") as report:
    loaded = load_bel_jsonl(gold_path, strict=False, progress_callback=report)
samples = list(loaded.samples)
if MAX_SAMPLES is not None:
    if isinstance(MAX_SAMPLES, bool) or not isinstance(MAX_SAMPLES, int) or MAX_SAMPLES < 1:
        raise ValueError("MAX_SAMPLES must be None or a positive integer.")
    samples = samples[:MAX_SAMPLES]

log(
    f"Loaded {len(loaded.samples):,} mention samples "
    f"({len(samples):,} selected) in {time.time() - started:.1f}s."
)
if loaded.issues:
    log(f"WARNING: {len(loaded.issues):,} data issue(s); first: {loaded.issues[0]}")
if not samples:
    raise ValueError(f"{gold_path} produced no usable mention samples.")

## Stage 6 — Configure the Retrieval Pipeline

`PipelineConfig` is validated on construction (fusion kind, `n_syn` in
{1, 3, 5, 10}, `top_k_concept` in 1..10). Feedback generation (`def`/`syn`/`std`)
is part of retrieval: failures degrade to `Q_0` instead of raising
(architecture §6). No re-ranking happens in this notebook: `rerank="top1"` only
fills `Response.cui` with the ranking's first entry. The saved candidates are
re-ranked separately by `reranking.ipynb`.

The feedback client is built from `LLM_BACKEND`: `"local"` loads
`LLM_MODEL_LOCAL` through `transformers` (weights are downloaded once, then
served offline from `HF_HOME`), `"api"` uses the OpenAI-compatible HTTP client.
`config.llm_model` records which one for provenance.

Re-run this cell after changing any constant or re-running an upstream cell:
stages 7–9 reuse the `linker` built here.

In [ ]:
config = PipelineConfig(
    encoder=ENCODER_MODEL,
    feedback_types=tuple(FEEDBACK_TYPES),
    n_syn=N_SYN,
    target_kb=TARGET_KB,
    top_k_retrieval=TOP_K_RETRIEVAL,
    top_k_concept=TOP_K_CONCEPT,
    fusion=FUSION,
    rerank="top1",              # no re-rank: the ranking's first entry; reranking.ipynb re-ranks
    llm_model=LLM_ACTIVE_MODEL,
    index_kb=INDEX_KB,
)

llm = None
if config.feedback_types:
    log(
        f"Feedback LLM: backend={LLM_BACKEND} model={LLM_ACTIVE_MODEL!r} "
        f"(weights load lazily on the first call)."
    )
    llm = build_llm_client(LLM_BACKEND, LLM_ACTIVE_MODEL)

feedback_generator = FeedbackGenerator(llm) if config.feedback_types else None

linker = EntityLinker(
    index,
    encoder,
    config,
    feedback_generator=feedback_generator,
    concept_names=concept_names,
    use_context_query=USE_CONTEXT_QUERY,
)

log(
    f"Retrieval pipeline ready: {len(index):,} index entries, "
    f"{len(concept_names):,} concepts; "
    f"feedback={config.feedback_types or 'none (baseline)'} "
    f"n_syn={config.n_syn} fusion={config.fusion}."
)

## Stage 7 — Run Candidate Retrieval

`link_batch` runs one batched pass per phase. Every request's `Q_0` is
encoded in **one forward pass** (`encode_mention_in_context`, or
`encoder.encode(mention)` when `USE_CONTEXT_QUERY = False`); feedback
generation is the only per-mention phase and its mentions run
**concurrently**; every feedback text of the batch is then embedded in one
call, fused in a single contraction and scanned with one batched cosine
product `R = Q @ A.T`, top-k along each row, on GPU when `use_gpu()` is
active. One pipeline pass produces, per mention, the truncated candidate list
saved for the next stage and the untruncated concept ranking.

This notebook **stops at the ranking**: `rerank="top1"` only copies the
ranking's first entry into `Response.cui`, so no re-ranking work happens
here. The saved candidates are re-ranked later, by `reranking.ipynb`.

Each phase reports through `progress_callback(phase, done, total)`, so the
run draws **one bar per phase** instead of a single bar that mixes feedback
generation with retrieval:

| bar | unit | what is happening |
| --- | --- | --- |
| `encode Q0` | encoder call | `Q_0` for every mention, one forward |
| `feedback` | mention | LLM round trips, the slow phase |
| `encode feedback` | encoder call | every feedback text, one forward |
| `retrieval` | index scan | one batched scan (RRF puts `n_i + 1` rows per mention in it) |
| `aggregation` | mention | surface hits → concept ranking |
| `assembly` | mention | truncate to `top_k_concept`, build the `Response` |


In [ ]:
requests = make_requests(samples, config)
with phase_progress() as report:
    started = time.time()
    responses = linker.link_batch(requests, progress_callback=report)

log(f"Retrieved candidates for {len(responses):,} mentions in {time.time() - started:.1f}s.")
empty = sum(1 for response in responses if not response.candidates)
if empty:
    log(f"WARNING: {empty:,} mentions produced no candidates.")

## Stage 8 — Save Candidates to One File

`save_candidates` writes one JSONL record per mention — sample fields, the
retrieved candidates, the untruncated concept ranking and retrieval metadata —
to the single configured `CANDIDATES_OUTPUT_PATH`. `reranking.ipynb` reads only this
file; it does not load the encoder or the vector index again. Both the save and
the verification reload report per-record progress.

The file is reloaded immediately to prove it is usable.

In [ ]:
log(f"Saving candidates to {candidates_path} ...")
started = time.time()
with progress_bar(len(samples), "Saving candidate records", "record") as report:
    save_candidates(candidates_path, samples, responses, progress_callback=report)
log(
    f"Saved {len(responses):,} records "
    f"({candidates_path.stat().st_size / 1024 ** 2:.2f} MB) "
    f"in {time.time() - started:.1f}s."
)

log("Reloading the candidate file to verify it ...")
with progress_bar(None, "Reloading candidate records", "record") as report:
    reloaded_samples, reloaded_responses = load_candidates(
        candidates_path, progress_callback=report
    )
assert len(reloaded_samples) == len(samples), (len(reloaded_samples), len(samples))
assert len(reloaded_responses) == len(responses), (len(reloaded_responses), len(responses))
assert [sample.gold_cui for sample in reloaded_samples] == [
    sample.gold_cui for sample in samples
]
assert [response.candidates for response in reloaded_responses] == [
    response.candidates for response in responses
]
assert [
    response.debug.get("concept_ranking") or []
    for response in reloaded_responses
] == [
    response.debug.get("concept_ranking") or []
    for response in responses
]
log(f"Verified: {len(reloaded_samples):,} samples and responses reloaded.")

## Stage 9 — Retrieval and Candidate Accuracy

- **R@1/R@5** from the untruncated concept ranking, before any re-ranking
  (architecture §10),
- **acc@1/acc@5** = share of mentions whose gold CUI is among the first k
  saved candidates: `acc@k = (1/n) Σ 1[gold_i ∈ top-k candidates_i]`.

No re-ranking happens here, so acc@k equals R@k for k ≤ `TOP_K_CONCEPT`.
`reranking.ipynb` reports acc@k again on the re-ranked candidate list, so a
retrieval-only run never claims E2E results.

In [ ]:
overall, by_label = metrics_from_responses(samples, responses)
acc1_overall, acc1_by_label = accuracy_at_k(samples, responses, 1)
acc5_overall, acc5_by_label = accuracy_at_k(samples, responses, 5)


def _row(label, metrics, acc1, acc5) -> str:
    return (
        f"{str(label):<16}{metrics.n:>7}"
        f"{100 * metrics.recall_at_1:>10.2f}{100 * metrics.recall_at_5:>10.2f}"
        f"{100 * acc1:>10.2f}{100 * acc5:>10.2f}"
    )


print(
    f"{'label':<16}{'n':>7}{'R@1 %':>10}{'R@5 %':>10}"
    f"{'acc@1 %':>10}{'acc@5 %':>10}"
)
print(_row("ALL", overall, acc1_overall, acc5_overall))
for label, metrics in by_label.items():
    print(_row(label, metrics, acc1_by_label[label], acc5_by_label[label]))

log("Candidate retrieval complete. Next: source/notebooks/GRF/reranking.ipynb")

## Stage 10 — Release Memory

Candidates are saved and the metrics are printed, so release both GPU
residents: the index's GPU copy (`release_gpu`), and the feedback model's
weights when a local backend was used (`TransformersChatClient.unload`). Then
drop the encoder, KB concepts and request list and empty the CUDA cache.
`samples` and `responses` stay for the metrics above.
Re-run stages 3–9 if you need the live pipeline again.

In [ ]:
if "index" in globals():
    index.release_gpu()
if "llm" in globals() and llm is not None and hasattr(llm, "unload"):
    llm.unload()
    log("Local feedback model weights unloaded.")

for name in (
    "linker",
    "feedback_generator",
    "llm",
    "index",
    "encoder",
    "kb_concepts",
    "concept_names",
    "requests",
    "reloaded_samples",
    "reloaded_responses",
):
    globals().pop(name, None)
gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free_b, total_b = torch.cuda.mem_get_info()
        log(
            f"CUDA cache emptied: {free_b / 1024 ** 3:.2f}/"
            f"{total_b / 1024 ** 3:.2f} GiB free."
        )
except ImportError:
    pass

log("Retrieval memory released (index, encoder, KB names, requests).")